## Section 0: Imports and Setup

Run this section first. It contains all notebook imports, project paths, and reusable helper imports used by the later sections.

In [19]:
import importlib
import os
import socket
import sys
from datetime import date
from pathlib import Path

import pandas as pd
import polars as pl
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.dialects import registry as _sa_registry

try:
    import ibm_db
    import ibm_db_dbi
    import ibm_db_sa
    DB2_DRIVER_READY = True
    DB2_DRIVER_ERROR = None
except Exception as exc:
    ibm_db = None
    ibm_db_dbi = None
    ibm_db_sa = None
    DB2_DRIVER_READY = False
    DB2_DRIVER_ERROR = exc
    print(f"DB2 driver is not ready: {type(exc).__name__}: {exc}")

_sa_registry.register("db2", "ibm_db_sa.ibm_db", "DB2Dialect_ibm_db")
_sa_registry.register("db2.ibm_db", "ibm_db_sa.ibm_db", "DB2Dialect_ibm_db")

load_dotenv()

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
for directory in (RAW_DIR, PROCESSED_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

import src.data_clean as data_clean
import src.data_enrich as data_enrich
import src.data_inspect as data_inspect
import src.data_model as data_model
from src.pull_data import TABLES, pull_and_save

importlib.reload(data_clean)
importlib.reload(data_enrich)
importlib.reload(data_inspect)
importlib.reload(data_model)

from src.data_clean import clean_raw_tables
from src.data_enrich import enrich_tables
from src.data_model import (
    check_flight_dashboard_joins,
    make_flight_dashboard_table,
)
from src.data_inspect import (
    RAW_TABLES,
    check_raw_files,
    load_tables,
    missing_value_counts,
    summary_statistics,
)

TABLE_NAMES = list(RAW_TABLES)

print(f"Polars version: {pl.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")

Polars version: 1.41.2
Project root: c:\Users\rothl\Desktop\ATT Group Project\ATT-Group8
Raw data: c:\Users\rothl\Desktop\ATT Group Project\ATT-Group8\data\raw
Processed data: c:\Users\rothl\Desktop\ATT Group Project\ATT-Group8\data\processed


In [20]:
DB_HOST = os.getenv("DB_HOST", "52.211.123.34")
DB_PORT = int(os.getenv("DB_PORT", "25010"))
DB_NAME = os.getenv("DB_NAME", "ATTPLANE")
DB_USERNAME = os.getenv("DB_USERNAME", "attgrp8")   # ← change to your group number
DB_PASSWORD = os.getenv("DB_PASSWORD", "bigdata")

print({"host": DB_HOST, "port": DB_PORT, "database": DB_NAME, "username": DB_USERNAME, "password": "***"})

{'host': '52.211.123.34', 'port': 25010, 'database': 'ATTPLANE', 'username': 'attgrp8', 'password': '***'}


In [21]:
def tcp_check(host: str = DB_HOST, port: int = DB_PORT, timeout: float = 8.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        sock.connect((host, port))
    return True

try:
    tcp_check()
    print(f"TCP connection succeeded: {DB_HOST}:{DB_PORT} is reachable")
except Exception as exc:
    print(f"TCP connection failed: {type(exc).__name__}: {exc}")

TCP connection succeeded: 52.211.123.34:25010 is reachable


In [22]:
def _make_raw_connection():
    if not DB2_DRIVER_READY:
        raise RuntimeError("DB2 driver is not ready") from DB2_DRIVER_ERROR

    conn_str = (
        f"HOSTNAME={DB_HOST};PORT={DB_PORT};DATABASE={DB_NAME};"
        f"PROTOCOL=TCPIP;UID={DB_USERNAME};PWD={DB_PASSWORD};"
        f"AUTHENTICATION=SERVER;CURRENTSCHEMA={DB_USERNAME.upper()};"
    )
    return ibm_db_dbi.Connection(ibm_db.connect(conn_str, "", ""))

def make_db2_engine():
    return create_engine("ibm_db_sa://", creator=_make_raw_connection)

engine = make_db2_engine()
engine

Engine(ibm_db_sa://)

In [23]:
def test_db_connection(engine) -> bool:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1 AS ok FROM SYSIBM.SYSDUMMY1"))
        row = result.fetchone()
    return row is not None and row[0] == 1

try:
    assert test_db_connection(engine)
    print("DB2 connection successful")
except Exception as exc:
    print("DB2 connection failed")
    print(f"{type(exc).__name__}: {exc}")
    print("\nChecklist:")
    print("1. Did the TCP check succeed?")
    print("2. Are ibm_db and ibm_db_sa installed in this notebook kernel?")
    print("3. Are DB_USERNAME and DB_PASSWORD correct for your group?")
    print("4. Is DB_NAME exactly ATTPLANE?")

DB2 connection successful


## Task 2: Data Loading

This section checks the raw parquet files in `data/raw/`. If files are missing, you can pull them from DB2 with `src.pull_data`, but the cleaning work below can use any raw files that are already present.

### 2.1 Pull Data From DB2 Using `src.pull_data`

This is the notebook hook into the existing data-pulling module. It calls `pull_and_save()` from `src/pull_data.py` and writes raw parquet files into `data/raw/`.

Leave `PULL_FROM_DB2 = False` for normal notebook runs. Change it to `True` only when you intentionally want to connect to DB2 and download missing raw files.

In [24]:
PULL_FROM_DB2 = False

if PULL_FROM_DB2:
    pulled_paths = pull_and_save(output_dir=RAW_DIR)
    pull_summary = pl.DataFrame(
        [
            {"table": table.lower(), "path": str(path)}
            for table, path in pulled_paths.items()
        ]
    )
else:
    pull_summary = pl.DataFrame(
        [
            {"table": table, "path": "not pulled in this run"}
            for table in TABLE_NAMES
        ]
    )

pull_summary

table,path
str,str
"""airplanes""","""not pulled in this run"""
"""airports""","""not pulled in this run"""
"""flights""","""not pulled in this run"""
"""passengers""","""not pulled in this run"""
"""routes""","""not pulled in this run"""
"""tickets""","""not pulled in this run"""


In [25]:
print(f"Raw data folder: {RAW_DIR}")
print(f"Expected tables: {', '.join(TABLE_NAMES)}")

Raw data folder: c:\Users\rothl\Desktop\ATT Group Project\ATT-Group8\data\raw
Expected tables: airplanes, airports, flights, passengers, routes, tickets


In [26]:
raw_status = check_raw_files(raw_dir=RAW_DIR, table_names=RAW_TABLES)
raw_status

table,status,rows,columns,issue
str,str,i64,i64,str
"""airplanes""","""ok""",796,13,null
"""airports""","""ok""",30,9,null
"""flights""","""ok""",1758638,10,null
"""passengers""","""ok""",500000,11,null
"""routes""","""ok""",624,7,null
"""tickets""","""needs_repull""",null,null,"""ComputeError: parquet: File ou…"


## Task 3: Inspect and Clean Raw Data

This section keeps inspection simple: first display the raw tables, then show summary statistics, then count null/missing values. Cleaning happens after that.

### 3.1 Raw Tables

Check which raw parquet files can be read. Then load a sample of each readable table and display the first few rows.

In [27]:
raw_file_health = check_raw_files(raw_dir=RAW_DIR, table_names=RAW_TABLES)
raw_file_health

table,status,rows,columns,issue
str,str,i64,i64,str
"""airplanes""","""ok""",796,13,null
"""airports""","""ok""",30,9,null
"""flights""","""ok""",1758638,10,null
"""passengers""","""ok""",500000,11,null
"""routes""","""ok""",624,7,null
"""tickets""","""needs_repull""",null,null,"""ComputeError: parquet: File ou…"


In [28]:
raw_tables = load_tables(
    data_dir=RAW_DIR,
    table_names=RAW_TABLES,
    sample_rows=10_000,
    skip_errors=True,
)

for table_name, df in raw_tables.items():
    print(f"{table_name}: showing 5 rows from {df.height:,} loaded rows")
    display(df.head(5))

Skipping tickets: ComputeError: parquet: File out of specification: The file must end with PAR1
airplanes: showing 5 rows from 796 loaded rows


aircraft_registration,model,seats_business,seats_premium,seats_economy,crew_members,build_date,fuel_gallons_hour,maintenance_last_acheck,maintenance_last_bcheck,maintenance_takeoffs,maintenance_flight_hours,total_flight_distance
str,str,f64,f64,f64,i64,date,i64,date,date,i64,i64,i64
"""IE53571""","""BOMBARDIER CRJ-1000""",null,null,100.0,8,2010-11-14,291,2024-04-11,2024-05-27,410,2487,931628
"""IE53730""","""BOMBARDIER CRJ-900""",90.0,null,null,4,2010-11-20,222,2024-04-16,2024-04-06,467,2914,555855
"""IE53994""","""AIRBUS A321-200(321)""",null,null,212.0,12,2010-11-25,386,2024-05-04,2024-03-31,213,1834,909017
"""IE54017""","""AIRBUS A350-900(359)""",31.0,24.0,293.0,15,2010-12-26,540,2024-03-08,2024-03-10,623,594,37905
"""IE54199""","""AIRBUS A330-200(332)""",null,21.0,293.0,15,2010-12-26,540,2024-05-28,2024-05-23,392,2030,819120


airports: showing 5 rows from 30 loaded rows


iata_code,airport,city,country,continent,timezone,latitude,longitude,airport_tax
str,str,str,str,str,str,f64,f64,f64
"""ADJ""","""Amman-Marka International Airp…","""Amman""","""JORDAN""","""ASIA""","""CUT+2""",31.9727,35.9916,9.65
"""ATL""","""Hartsfield Jackson Atlanta""","""Atlanta""","""UNITED STATES""","""AMERICA""","""CUT-5""",33.6367,-84.428101,0.82
"""BCN""","""Barcelona International Airpor…","""Barcelona""","""SPAIN""","""EUROPE""","""CUT+1""",41.2971,2.07846,11.79
"""BLQ""","""Bologna Guglielmo Marconi Airp…","""Bologna""","""ITALY""","""EUROPE""","""CUT+1""",44.5354,11.2887,null
"""BOG""","""El Dorado International Airpor…","""Bogota""","""COLOMBIA""","""AMERICA""","""CUT+2""",4.70159,-74.1469,null


flights: showing 5 rows from 10,000 loaded rows


flight_id,flight_leg,frequency,route_code,departure,arrival,airplane,price_economy,price_premium,price_business
str,i64,str,str,datetime[ns],datetime[ns],str,f64,f64,f64
"""IE0003""",0,"""F1""","""R235""",2010-11-08 06:45:00,2010-11-08 07:31:00,"""IE06307""",41.68,57.71,84.18
"""IE0003""",0,"""F1""","""R235""",2010-11-09 06:45:00,2010-11-09 07:31:00,"""IE06307""",42.34,60.03,79.44
"""IE0003""",0,"""F1""","""R235""",2010-11-10 06:45:00,2010-11-10 07:31:00,"""IE06307""",50.94,66.36,85.46
"""IE0003""",0,"""F1""","""R235""",2010-11-11 06:45:00,2010-11-11 07:31:00,"""IE06307""",40.24,69.92,80.55
"""IE0003""",0,"""F1""","""R235""",2010-11-12 06:45:00,2010-11-12 07:31:00,"""IE06307""",51.32,63.59,76.14


passengers: showing 5 rows from 10,000 loaded rows


id,firstnme,midinit,lastname,gender,birth_date,passport,country,vipcard,phone,email
i64,str,str,str,str,date,str,str,str,str,str
1,"""Ivanna""","""N""","""Barrow""","""F""",1978-02-27,"""317088427""","""JORDAN""",null,"""931916171""",null
2,"""Cecelia""","""L""","""Yeomans""","""F""",1976-07-27,"""308948657""","""UNITED STATES""",null,null,"""cecelia.yeomans@gmail.com"""
3,"""Mila""",""" ""","""Stafford""","""F""",2010-12-13,"""499306172""","""SPAIN""",null,null,"""mila.stafford@gmail.com"""
4,"""Zane""",""" ""","""Piper""","""M""",1957-05-01,"""199026136""","""SPAIN""",null,"""281717284""","""zane.piper@gmail.com"""
5,"""Colton""","""O""","""Yeates""","""M""",2010-12-14,"""508765552""","""SPAIN""","""VIP-847006""",null,"""colton.yeates@gmail.com"""


routes: showing 5 rows from 624 loaded rows


route_code,origin,destination,parent_route,leg_number,distance,flight_minutes
str,str,str,str,i64,i64,i64
"""R001""","""JFK""","""TPA""","""R001 """,0,1621,135
"""R002""","""TPA""","""ATL""","""R001 """,1,655,66
"""R003""","""ATL""","""JFK""","""R001 """,2,1223,107
"""R004""","""JFK""","""LAS""","""R004 """,0,3613,278
"""R005""","""LAS""","""ATL""","""R004 """,1,2807,220


### 3.2 Summary Statistics

Show simple numeric statistics for each readable raw table.

In [29]:
raw_summary_statistics = summary_statistics(raw_tables)
raw_summary_statistics

table,column,min,mean,median,max
str,str,f64,f64,f64,f64
"""airplanes""","""crew_members""",4.0,10.507538,12.0,15.0
"""airplanes""","""fuel_gallons_hour""",222.0,375.295226,386.0,540.0
"""airplanes""","""maintenance_flight_hours""",5.0,1493.090452,1484.0,2993.0
"""airplanes""","""maintenance_takeoffs""",0.0,495.664573,494.5,999.0
"""airplanes""","""seats_business""",16.0,34.146199,31.0,90.0
…,…,…,…,…,…
"""flights""","""price_premium""",26.42,82.118466,84.025,166.11
"""passengers""","""id""",1.0,5203.3445,5000.5,299777.0
"""routes""","""distance""",176.0,6549.858974,7633.0,19348.0


### 3.3 Null and Missing Value Counts

For this notebook, missing values means Polars nulls plus blank strings in text columns.

In [30]:
raw_missing_values = missing_value_counts(raw_tables)
raw_missing_values

table,column,dtype,rows,null_count,blank_string_count,missing_count,missing_pct
str,str,str,i64,i64,i64,i64,f64
"""airplanes""","""seats_premium""","""Float64""",796,469,0,469,0.589196
"""airplanes""","""seats_business""","""Float64""",796,112,0,112,0.140704
"""airplanes""","""seats_economy""","""Float64""",796,70,0,70,0.08794
"""airplanes""","""aircraft_registration""","""String""",796,0,0,0,0.0
"""airplanes""","""build_date""","""Date""",796,0,0,0,0.0
…,…,…,…,…,…,…,…
"""routes""","""flight_minutes""","""Int64""",624,0,0,0,0.0
"""routes""","""leg_number""","""Int64""",624,0,0,0,0.0
"""routes""","""origin""","""String""",624,0,0,0,0.0


### 3.4 Cleaning Rules

The cleaning module is intentionally small. It applies simple rules:

- keep the original column names
- trim text columns
- turn blanks and placeholders like `NA`, `N/A`, and `NULL` into real nulls
- parse dates only when you pass date columns to the function
- remove rows that are exact duplicates

The module itself does not hardcode ATTPLANE table names or column names. In the notebook cell below, we pass the date columns we want parsed before enrichment.

In [31]:
date_columns_by_table = {
    "airplanes": (
        "build_date",
        "maintenance_last_acheck",
        "maintenance_last_bcheck",
    ),
    "passengers": ("birth_date",),
}

datetime_columns_by_table = {
    "flights": ("departure", "arrival"),
    "tickets": ("departure",),
}

cleaned_tables = clean_raw_tables(
    raw_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
    date_columns_by_table=date_columns_by_table,
    datetime_columns_by_table=datetime_columns_by_table,
    skip_errors=True,
)

processed_summary = pl.DataFrame(
    [
        {
            "table": table_name,
            "rows": df.height,
            "columns": df.width,
            "processed_path": str(PROCESSED_DIR / f"{table_name}_clean.parquet"),
        }
        for table_name, df in cleaned_tables.items()
    ]
).sort("table")

processed_summary

Skipping tickets: ComputeError: parquet: File out of specification: The file must end with PAR1


table,rows,columns,processed_path
str,i64,i64,str
"""airplanes""",796,13,"""c:\Users\rothl\Desktop\ATT Gro…"
"""airports""",30,9,"""c:\Users\rothl\Desktop\ATT Gro…"
"""flights""",1758638,10,"""c:\Users\rothl\Desktop\ATT Gro…"
"""passengers""",500000,11,"""c:\Users\rothl\Desktop\ATT Gro…"
"""routes""",624,7,"""c:\Users\rothl\Desktop\ATT Gro…"


### 3.5 Check Cleaned Output

Load the cleaned parquet files and check their null/missing value counts.

In [32]:
processed_table_names = tuple(
    path.name.removesuffix(".parquet")
    for path in sorted(PROCESSED_DIR.glob("*_clean.parquet"))
)

processed_tables = load_tables(
    data_dir=PROCESSED_DIR,
    table_names=processed_table_names,
    skip_errors=True,
)

processed_missing_values = missing_value_counts(processed_tables)
processed_missing_values

table,column,dtype,rows,null_count,blank_string_count,missing_count,missing_pct
str,str,str,i64,i64,i64,i64,f64
"""airplanes_clean""","""seats_premium""","""Float64""",796,469,0,469,0.589196
"""airplanes_clean""","""seats_business""","""Float64""",796,112,0,112,0.140704
"""airplanes_clean""","""seats_economy""","""Float64""",796,70,0,70,0.08794
"""airplanes_clean""","""aircraft_registration""","""String""",796,0,0,0,0.0
"""airplanes_clean""","""build_date""","""Date""",796,0,0,0,0.0
…,…,…,…,…,…,…,…
"""routes_clean""","""flight_minutes""","""Int64""",624,0,0,0,0.0
"""routes_clean""","""leg_number""","""Int64""",624,0,0,0,0.0
"""routes_clean""","""origin""","""String""",624,0,0,0,0.0


## Task 4: Enrich and Model Tables

This section adds simple business features to each table first, then joins the flight-related tables into one analysis-ready master table.

Saving rule for this project:

- save cleaned tables as `*_clean.parquet` checkpoints
- keep enriched intermediate tables in memory
- save the final joined table as `master_flight_dashboard.parquet`

### 4.1 Enrich Each Table

Use the cleaned tables from Task 3 if they are already in memory. Otherwise, read the saved `*_clean.parquet` files from `data/processed`.

In [33]:
if "cleaned_tables" in globals():
    tables_for_enrichment = cleaned_tables
else:
    tables_for_enrichment = {
        path.name.removesuffix("_clean.parquet"): pl.read_parquet(path)
        for path in sorted(PROCESSED_DIR.glob("*_clean.parquet"))
    }

enriched_tables = enrich_tables(
    tables_for_enrichment,
    reference_date=date.today(),
)

enriched_summary = pl.DataFrame(
    [
        {
            "table": table_name,
            "rows": df.height,
            "columns": df.width,
        }
        for table_name, df in enriched_tables.items()
    ]
).sort("table")

enriched_summary

table,rows,columns
str,i64,i64
"""airplanes""",796,17
"""airports""",30,12
"""flights""",1758638,17
"""passengers""",500000,14
"""routes""",624,11


### 4.2 Preview Enriched Features

Look at a few new fields that could be useful for a dashboard.

In [34]:
if "flights" in enriched_tables:
    flight_columns = [
        column
        for column in (
            "flight_id",
            "departure_date",
            "departure_hour",
            "is_weekend",
            "scheduled_duration_minutes",
        )
        if column in enriched_tables["flights"].columns
    ]
    display(enriched_tables["flights"].select(flight_columns).head(5))

if "routes" in enriched_tables:
    route_columns = [
        column
        for column in (
            "route_code",
            "distance",
            "distance_band",
            "flight_hours",
            "average_speed",
        )
        if column in enriched_tables["routes"].columns
    ]
    display(enriched_tables["routes"].select(route_columns).head(5))

if "airplanes" in enriched_tables:
    airplane_columns = [
        column
        for column in (
            "aircraft_registration",
            "model",
            "model_family",
            "total_seats",
        )
        if column in enriched_tables["airplanes"].columns
    ]
    display(enriched_tables["airplanes"].select(airplane_columns).head(5))

flight_id,departure_date,departure_hour,is_weekend,scheduled_duration_minutes
str,date,i8,bool,i64
"""IE0003""",2010-11-08,6,false,46
"""IE0003""",2010-11-09,6,false,46
"""IE0003""",2010-11-10,6,false,46
"""IE0003""",2010-11-11,6,false,46
"""IE0003""",2010-11-12,6,false,46


route_code,distance,distance_band,flight_hours,average_speed
str,i64,str,f64,f64
"""R001""",1621,"""medium""",2.25,720.4
"""R002""",655,"""short""",1.1,595.5
"""R003""",1223,"""short""",1.783333,685.8
"""R004""",3613,"""medium""",4.633333,779.8
"""R005""",2807,"""medium""",3.666667,765.5


aircraft_registration,model,model_family,total_seats
str,str,str,f64
"""IE53571""","""BOMBARDIER CRJ-1000""","""Other""",100.0
"""IE53730""","""BOMBARDIER CRJ-900""","""Other""",90.0
"""IE53994""","""AIRBUS A321-200(321)""","""Airbus""",212.0
"""IE54017""","""AIRBUS A350-900(359)""","""Airbus""",348.0
"""IE54199""","""AIRBUS A330-200(332)""","""Airbus""",314.0


### 4.3 Join Tables, Verify Row Counts, and Save Master

Build one flight-level dashboard table by joining flights to routes, airplanes, and origin/destination airports. Then check that the joins did not unexpectedly add or lose rows.

This master table does not include `tickets` yet because the tickets parquet file is corrupted. It also does not include passenger fields yet, because passengers connect to flights through tickets.

In [35]:
required_join_tables = {"flights", "routes", "airplanes", "airports"}
missing_join_tables = required_join_tables - set(enriched_tables)

if missing_join_tables:
    print(f"Missing tables for flight dashboard join: {sorted(missing_join_tables)}")
else:
    master_flight_dashboard = make_flight_dashboard_table(
        enriched_tables["flights"],
        enriched_tables["routes"],
        enriched_tables["airplanes"],
        enriched_tables["airports"],
    )

    join_checks = check_flight_dashboard_joins(
        enriched_tables["flights"],
        enriched_tables["routes"],
        enriched_tables["airplanes"],
        enriched_tables["airports"],
        master_flight_dashboard,
    )

    master_table_path = PROCESSED_DIR / "master_flight_dashboard.parquet"
    master_flight_dashboard.write_parquet(master_table_path)

    display(join_checks)

    print(f"Flight rows before join: {enriched_tables['flights'].height:,}")
    print(f"Master dashboard rows after join: {master_flight_dashboard.height:,}")
    print(f"Saved master table to: {master_table_path}")

join,left_rows,joined_rows,unmatched_rows,duplicate_right_keys,row_count_ok,all_keys_matched
str,i64,i64,i64,i64,bool,bool
"""flights to routes""",1758638,1758638,0,0,true,true
"""flights to airplanes""",1758638,1758638,0,0,true,true
"""routes to origin airports""",624,624,0,0,true,true
"""routes to destination airports""",624,624,0,0,true,true
"""final flight dashboard table""",1758638,1758638,null,null,true,null


Flight rows before join: 1,758,638
Master dashboard rows after join: 1,758,638
Saved master table to: c:\Users\rothl\Desktop\ATT Group Project\ATT-Group8\data\processed\master_flight_dashboard.parquet


In [36]:
master_path = PROCESSED_DIR / "master_flight_dashboard.parquet"

if "master_flight_dashboard" not in globals():
    master_flight_dashboard = pl.read_parquet(master_path)

display(master_flight_dashboard.head(10))

flight_id,flight_leg,frequency,route_code,departure,arrival,airplane,price_economy,price_premium,price_business,departure_date,departure_year,departure_month,departure_weekday,departure_hour,is_weekend,scheduled_duration_minutes,route_route_code,route_origin,route_destination,route_parent_route,route_leg_number,route_distance,route_flight_minutes,route_distance_band,route_flight_hours,route_average_speed,route_has_parent_route,airplane_aircraft_registration,airplane_model,airplane_seats_business,airplane_seats_premium,airplane_seats_economy,airplane_crew_members,airplane_build_date,airplane_fuel_gallons_hour,airplane_maintenance_last_acheck,airplane_maintenance_last_bcheck,airplane_maintenance_takeoffs,airplane_maintenance_flight_hours,airplane_total_flight_distance,airplane_total_seats,airplane_has_business_class,airplane_has_premium_class,airplane_model_family,origin_iata_code,origin_airport,origin_city,origin_country,origin_continent,origin_timezone,origin_latitude,origin_longitude,origin_airport_tax,origin_airport_label,origin_has_coordinates,origin_airport_tax_band,destination_iata_code,destination_airport,destination_city,destination_country,destination_continent,destination_timezone,destination_latitude,destination_longitude,destination_airport_tax,destination_airport_label,destination_has_coordinates,destination_airport_tax_band,is_domestic_route,is_same_continent_route
str,i64,str,str,datetime[μs],datetime[μs],str,f64,f64,f64,date,i32,i8,i8,i8,bool,i64,str,str,str,str,i64,i64,i64,str,f64,f64,bool,str,str,f64,f64,f64,i64,date,i64,date,date,i64,i64,i64,f64,bool,bool,str,str,str,str,str,str,str,f64,f64,f64,str,bool,str,str,str,str,str,str,str,f64,f64,f64,str,bool,str,bool,bool
"""IE0003""",0,"""F1""","""R235""",2010-11-08 06:45:00,2010-11-08 07:31:00,"""IE06307""",41.68,57.71,84.18,2010-11-08,2010,11,1,6,false,46,"""R235""","""MAD""","""GRX""","""R235""",0,365,46,"""short""",0.766667,476.1,true,"""IE06307""","""BOMBARDIER CRJ-200""",50.0,null,null,4,2000-10-24,222,2024-04-20,2024-05-25,974,470,925294,50.0,true,false,"""Other""","""MAD""","""Adolfo Suarez Madrid Barajas""","""Madrid""","""SPAIN""","""EUROPE""","""CUT+1""",40.471926,-3.56264,6.69,"""Madrid (MAD)""",true,"""low""","""GRX""","""Federico Garcia Lorca Airport""","""Granada""","""SPAIN""","""EUROPE""","""CUT+1""",37.188702,-3.77736,4.91,"""Granada (GRX)""",true,"""low""",true,true
"""IE0003""",0,"""F1""","""R235""",2010-11-09 06:45:00,2010-11-09 07:31:00,"""IE06307""",42.34,60.03,79.44,2010-11-09,2010,11,2,6,false,46,"""R235""","""MAD""","""GRX""","""R235""",0,365,46,"""short""",0.766667,476.1,true,"""IE06307""","""BOMBARDIER CRJ-200""",50.0,null,null,4,2000-10-24,222,2024-04-20,2024-05-25,974,470,925294,50.0,true,false,"""Other""","""MAD""","""Adolfo Suarez Madrid Barajas""","""Madrid""","""SPAIN""","""EUROPE""","""CUT+1""",40.471926,-3.56264,6.69,"""Madrid (MAD)""",true,"""low""","""GRX""","""Federico Garcia Lorca Airport""","""Granada""","""SPAIN""","""EUROPE""","""CUT+1""",37.188702,-3.77736,4.91,"""Granada (GRX)""",true,"""low""",true,true
"""IE0003""",0,"""F1""","""R235""",2010-11-10 06:45:00,2010-11-10 07:31:00,"""IE06307""",50.94,66.36,85.46,2010-11-10,2010,11,3,6,false,46,"""R235""","""MAD""","""GRX""","""R235""",0,365,46,"""short""",0.766667,476.1,true,"""IE06307""","""BOMBARDIER CRJ-200""",50.0,null,null,4,2000-10-24,222,2024-04-20,2024-05-25,974,470,925294,50.0,true,false,"""Other""","""MAD""","""Adolfo Suarez Madrid Barajas""","""Madrid""","""SPAIN""","""EUROPE""","""CUT+1""",40.471926,-3.56264,6.69,"""Madrid (MAD)""",true,"""low""","""GRX""","""Federico Garcia Lorca Airport""","""Granada""","""SPAIN""","""EUROPE""","""CUT+1""",37.188702,-3.77736,4.91,"""Granada (GRX)""",true,"""low""",true,true
"""IE0003""",0,"""F1""","""R235""",2010-11-11 06:45:00,2010-11-11 07:31:00,"""IE06307""",40.24,69.92,80.55,2010-11-11,2010,11,4,6,false,46,"""R235""","""MAD""","""GRX""","""R235""",0,365,46,"""short""",0.766667,476.1,true,"""IE06307""","""BOMBARDIER CRJ-200